# Taller Etiquetado de secuencias (POS/NER)
## Notebook base del equipo

**Equipo:** _(nombres)_ · **Ruta:** A (transferencia) / B (tipos propios) · **Problema:** _(una frase)_

Este notebook sigue las seis fases de CRISP-ML(Q) de la guía del taller. Trae las **herramientas** listas (carga de datos, métricas, modelos, evaluación); las **decisiones** son del equipo. Busquen las marcas:

- `# TODO` → algo que el equipo debe completar o decidir.
- **✍️ Responder** → celdas de texto que van al documento técnico.

Regla de oro: **ejecutar de arriba abajo** (Kernel → Restart & Run All) antes de entregar.

## 0 · Configuración del equipo

In [2]:
# TODO: completar antes de empezar
EQUIPO = "XX"                               # número o nombre corto del equipo
RUTA = "A"                                  # "A" = tipos CoNLL (PER, LOC, ORG, MISC) · "B" = tipos propios
TIPOS = ["PER", "LOC", "ORG", "MISC"]        # ruta B: p. ej. ["NORMA", "ENTIDAD", "FECHA"]

# Descripción de cada tipo (la usa el prompt del LLM y va en la guía de anotación)
DESCRIPCIONES = {
    "PER": "personas",
    "LOC": "lugares geográficos",
    "ORG": "organizaciones, instituciones, empresas, equipos",
    "MISC": "otras entidades con nombre propio: eventos, obras, nacionalidades, productos",
}

# Archivos (formato CoNLL: token<TAB>etiqueta, línea en blanco entre oraciones)
DATA_DIR = "data"
ARCHIVO_EQUIPO = f"{DATA_DIR}/equipo_{EQUIPO}.conll"      # versión consensuada
ARCHIVO_ANOT_A = f"{DATA_DIR}/equipo_{EQUIPO}_A.conll"    # anotador A (doble anotación)
ARCHIVO_ANOT_B = f"{DATA_DIR}/equipo_{EQUIPO}_B.conll"    # anotador B

# Criterios de éxito (copiarlos de la ficha del problema)
CRITERIO_F1 = 0.75          # F1 por entidad mínimo fuera de dominio
CRITERIO_SEG = 1.0          # segundos por oración máximos

# Modelos pesados (activar solo si el entorno lo permite)
RUN_TRANSFORMER = False     # BETO afinado (recomendado con GPU)
RUN_LLM = False             # Qwen2.5-1.5B-Instruct local, pocos ejemplos
SEED = 42

In [3]:
# Paquetes básicos (ejecutar una vez; si instala algo, reiniciar el kernel)
%pip install -q nltk python-crfsuite pandas matplotlib
# Solo si RUN_TRANSFORMER o RUN_LLM:
if RUN_TRANSFORMER or RUN_LLM:
    %pip install -q torch "transformers>=4.56" datasets accelerate

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os, re, json, time, random, platform
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(SEED); np.random.seed(SEED)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("salidas", exist_ok=True)

# Reproducibilidad: registrar versiones (va al documento)
from importlib.metadata import version, PackageNotFoundError
versiones = {"python": platform.python_version()}
for paquete in ["numpy", "pandas", "nltk", "python-crfsuite", "torch", "transformers", "datasets"]:
    try:
        versiones[paquete] = version(paquete)
    except PackageNotFoundError:
        versiones[paquete] = "no instalado"
versiones

{'python': '3.14.6',
 'numpy': '2.4.6',
 'pandas': '3.0.3',
 'nltk': '3.10.0',
 'python-crfsuite': '0.9.12',
 'torch': '2.14.0',
 'transformers': '5.17.0',
 'datasets': '5.0.1'}

## Herramientas comunes (no modificar)
Formato interno: cada oración es una lista de tokens y una lista de etiquetas BIO de igual longitud.

In [5]:
def read_conll(path):
    # Lee token<TAB>etiqueta; devuelve lista de (tokens, etiquetas)
    sents, toks, tags = [], [], []
    with open(path, encoding="utf-8") as f:
        for n, line in enumerate(f, 1):
            line = line.rstrip("\n")
            if not line.strip():
                if toks: sents.append((toks, tags))
                toks, tags = [], []
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                raise ValueError(f"{path}, línea {n}: se esperaba 'token<TAB>etiqueta' y se leyó {line!r}")
            toks.append(parts[0]); tags.append(parts[1].strip())
    if toks: sents.append((toks, tags))
    return sents

def write_conll(sents, path):
    with open(path, "w", encoding="utf-8") as f:
        for toks, tags in sents:
            for w, t in zip(toks, tags): f.write(f"{w}\t{t}\n")
            f.write("\n")

def bio_to_spans(labels):
    # BIO -> [(inicio, fin_exclusivo, tipo)]; un I- huérfano abre entidad (convención conlleval)
    spans, start, typ = [], None, None
    for i, lab in enumerate(list(labels) + ["O"]):
        if not (lab.startswith("I-") and typ == lab[2:]):
            if typ is not None: spans.append((start, i, typ))
            start, typ = None, None
            if lab[:2] in ("B-", "I-"): start, typ = i, lab[2:]
    return spans

def validate_bio(sents, tipos):
    # Devuelve problemas de formato: tipos desconocidos, I- sin B- previo
    problemas = []
    for k, (toks, tags) in enumerate(sents):
        prev = "O"
        for i, t in enumerate(tags):
            if t != "O" and (t[:2] not in ("B-", "I-") or t[2:] not in tipos):
                problemas.append((k, i, toks[i], t, "etiqueta desconocida"))
            elif t.startswith("I-") and prev[2:] != t[2:]:
                problemas.append((k, i, toks[i], t, "I- sin B- previo"))
            prev = t
    return pd.DataFrame(problemas, columns=["oración", "posición", "token", "etiqueta", "problema"])

def entity_prf(gold_seqs, pred_seqs):
    # P, R, F1 por entidad (coincidencia exacta de inicio, fin y tipo), por tipo y micro
    st = defaultdict(lambda: [0, 0, 0])
    for g, p in zip(gold_seqs, pred_seqs):
        G, P = set(bio_to_spans(g)), set(bio_to_spans(p))
        for s in G & P: st[s[2]][0] += 1
        for s in P - G: st[s[2]][1] += 1
        for s in G - P: st[s[2]][2] += 1
    def prf(tp, fp, fn):
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        return p, r, (2 * p * r / (p + r) if p + r else 0.0)
    rows = [(t, *prf(*st[t]), st[t][0] + st[t][2]) for t in sorted(st)]
    TP, FP, FN = (sum(v[i] for v in st.values()) for i in range(3))
    rows.append(("micro", *prf(TP, FP, FN), TP + FN))
    return pd.DataFrame(rows, columns=["tipo", "P", "R", "F1", "soporte"]).set_index("tipo")

def bootstrap_f1(gold, pred, B=1000, seed=SEED):
    # Intervalo del 95 % del F1 micro remuestreando oraciones
    rnd, n, vals = random.Random(seed), len(gold), []
    for _ in range(B):
        idx = [rnd.randrange(n) for _ in range(n)]
        vals.append(entity_prf([gold[i] for i in idx], [pred[i] for i in idx]).loc["micro", "F1"])
    vals.sort()
    return vals[int(0.025 * B)], vals[int(0.975 * B)]

def error_breakdown(sents, preds):
    # Clasifica cada error: tipo equivocado, frontera, omitida (FN), espuria (FP)
    filas = []
    for k, ((toks, gold), pred) in enumerate(zip(sents, preds)):
        G, P = set(bio_to_spans(gold)), set(bio_to_spans(pred))
        for s in G - P:
            mismo = [q for q in P if q[:2] == s[:2]]
            solape = [q for q in P if q[0] < s[1] and s[0] < q[1]]
            clase = "tipo equivocado" if mismo else ("frontera" if solape else "omitida (FN)")
            pr = (mismo or solape or [None])[0]
            filas.append((k, " ".join(toks[s[0]:s[1]]), s[2], clase,
                          "" if pr is None else f"{' '.join(toks[pr[0]:pr[1]])} [{pr[2]}]"))
        for s in P - G:
            if not any(q[0] < s[1] and s[0] < q[1] for q in G):
                filas.append((k, "", "", "espuria (FP)", f"{' '.join(toks[s[0]:s[1]])} [{s[2]}]"))
    return pd.DataFrame(filas, columns=["oración", "oro", "tipo_oro", "clase_error", "predicho"])

def resumen_datos(sents, nombre):
    ents = Counter(t for _, tags in sents for _, _, t in bio_to_spans(tags))
    return {"conjunto": nombre, "oraciones": len(sents), "tokens": sum(len(t) for t, _ in sents),
            "entidades": sum(ents.values()), **{f"n_{k}": v for k, v in sorted(ents.items())}}

print("Herramientas cargadas.")

Herramientas cargadas.


## Fase 1 · Entendimiento del negocio y de los datos

**✍️ Responder** (copiar al documento):

| Campo | Respuesta del equipo |
|---|---|
| Problema | |
| Usuario y decisión | |
| Objetivo de ML (tarea, entrada, salida, etiquetas) | |
| Éxito de negocio | |
| Éxito de ML (métrica y umbral) | |
| Éxito económico (costo, latencia) | |
| Datos (fuente, cantidad, licencia, fecha) | |
| Viabilidad | |

### Registro de riesgos
Probabilidad y gravedad de 1 a 3. Prioridad = probabilidad × gravedad; los de prioridad ≥ 6 necesitan mitigación antes de seguir. El registro se actualiza en cada fase.

In [6]:
# TODO: mínimo tres riesgos (uno lo aporta el otro equipo en la revisión cruzada)
riesgos = pd.DataFrame([
    {"fase": 1, "riesgo": "(describir)", "probabilidad": 2, "gravedad": 2, "mitigación": "(describir)", "evidencia": "(qué celda o tabla lo muestra)"},
])
riesgos["prioridad"] = riesgos.probabilidad * riesgos.gravedad
riesgos.sort_values("prioridad", ascending=False)

,fase,riesgo,probabilidad,gravedad,mitigación,evidencia,prioridad
0,1,(describir),2,2,(describir),(qué celda o tabla lo muestra),4


## Fase 2 · Preparación de datos

### 2.1 Acuerdo entre anotadores
Cada integrante anota **por separado** las mismas oraciones (`_A.conll`, `_B.conll`). Se mide el F1 por entidad tomando a A como referencia. Si es menor que 0.80, revisen las diferencias y ajusten la guía.

In [7]:
if os.path.exists(ARCHIVO_ANOT_A) and os.path.exists(ARCHIVO_ANOT_B):
    anot_A, anot_B = read_conll(ARCHIVO_ANOT_A), read_conll(ARCHIVO_ANOT_B)
    assert len(anot_A) == len(anot_B), "Los dos archivos deben tener las mismas oraciones"
    for (ta, _), (tb, _) in zip(anot_A, anot_B):
        assert ta == tb, f"Tokenización distinta: {ta[:6]} vs {tb[:6]}"
    acuerdo = entity_prf([t for _, t in anot_A], [t for _, t in anot_B])
    display(acuerdo)
    print(f"Acuerdo (F1 micro por entidad): {acuerdo.loc['micro', 'F1']:.3f}")
    # Oraciones con desacuerdo, para discutir y mejorar la guía
    desac = [(k, " ".join(ta), bio_to_spans(ga), bio_to_spans(gb))
             for k, ((ta, ga), (_, gb)) in enumerate(zip(anot_A, anot_B)) if bio_to_spans(ga) != bio_to_spans(gb)]
    print(f"Oraciones con desacuerdo: {len(desac)} de {len(anot_A)}")
    display(pd.DataFrame(desac, columns=["oración", "texto", "spans_A", "spans_B"]).head(10))
else:
    print("Aún no están los archivos de doble anotación:", ARCHIVO_ANOT_A, ARCHIVO_ANOT_B)

Aún no están los archivos de doble anotación: data/equipo_XX_A.conll data/equipo_XX_B.conll


### 2.2 Muestra consensuada del equipo
Si todavía no existe el archivo del equipo, el notebook usa una **muestra DEMO** de CoNLL-2002 para que todo corra. La DEMO no sirve para el informe.

In [8]:
import nltk
nltk.download("conll2002", quiet=True)
from nltk.corpus import conll2002

def load_conll2002(fileid):
    return [([w for w, _, _ in s], [t for _, _, t in s]) for s in conll2002.iob_sents(fileid) if s]

DEMO = not os.path.exists(ARCHIVO_EQUIPO)
if DEMO:
    print("⚠️  No se encontró", ARCHIVO_EQUIPO, "→ se usa una muestra DEMO (40 oraciones de CoNLL-2002 testb).")
    equipo = load_conll2002("esp.testb")[:40]
else:
    equipo = read_conll(ARCHIVO_EQUIPO)

problemas = validate_bio(equipo, TIPOS)
print(f"Problemas de formato: {len(problemas)}")
display(problemas.head(10))
pd.DataFrame([resumen_datos(equipo, "muestra del equipo")])

⚠️  No se encontró data/equipo_XX.conll → se usa una muestra DEMO (40 oraciones de CoNLL-2002 testb).
Problemas de formato: 0


,oración,posición,token,etiqueta,problema


,conjunto,oraciones,tokens,entidades,n_LOC,n_MISC,n_ORG,n_PER
0,muestra del equipo,40,1577,144,43,14,55,32


### 2.3 Particiones
- **Ruta A:** se entrena con CoNLL-2002 `esp.train`, se evalúa en dominio con `esp.testb` y **toda la muestra del equipo es la prueba fuera de dominio**.
- **Ruta B:** la muestra del equipo se divide 70/30 con semilla fija. **Nunca** se ajustan hiperparámetros mirando la prueba.

In [9]:
if RUTA == "A":
    train = load_conll2002("esp.train")
    test_dominio = [s for s in load_conll2002("esp.testb")[40 if DEMO else 0:]]
    test_fuera = equipo
else:
    idx = list(range(len(equipo))); random.Random(SEED).shuffle(idx)
    corte = int(0.7 * len(idx))
    train = [equipo[i] for i in idx[:corte]]
    test_dominio = None
    test_fuera = [equipo[i] for i in idx[corte:]]

ficha_datos = pd.DataFrame([resumen_datos(train, "entrenamiento")]
                           + ([resumen_datos(test_dominio, "prueba en dominio")] if test_dominio else [])
                           + [resumen_datos(test_fuera, "prueba del equipo")]).fillna(0)
ficha_datos

,conjunto,oraciones,tokens,entidades,n_LOC,n_MISC,n_ORG,n_PER
0,entrenamiento,8323,264715,18798,4914,2173,7390,4321
1,prueba en dominio,1477,49956,3415,1041,326,1345,703
2,prueba del equipo,40,1577,144,43,14,55,32


**✍️ Responder:** fuente y fecha de los textos, permiso o licencia, qué se excluyó y por qué, acuerdo antes y después de la discusión.

## Fase 3 · Modelado
Cada modelo se registra como una función `predecir(lista_de_tokens) -> lista_de_etiquetas` en el diccionario `MODELOS`. La fase 4 los evalúa todos igual.

In [10]:
MODELOS = {}

# --- 3.1 Línea base: etiqueta más frecuente por token
cuenta = defaultdict(Counter)
for toks, tags in train:
    for w, t in zip(toks, tags): cuenta[w][t] += 1
mas_frecuente = {w: c.most_common(1)[0][0] for w, c in cuenta.items()}

def predecir_base(sents_tokens):
    return [[mas_frecuente.get(w, "O") for w in toks] for toks in sents_tokens]

MODELOS["Etiqueta más frecuente"] = predecir_base
print("Línea base lista.")

Línea base lista.


### 3.2 CRF con un rasgo propio del dominio
El diccionario `GAZETTEER` agrega un rasgo `gaz:TIPO` cuando la palabra (en minúscula) está en la lista. **TODO:** llénenlo con conocimiento del dominio (barrios, entidades públicas, siglas, normas…) y midan su efecto con y sin él.

In [ ]:
import pycrfsuite

# TODO: listas del dominio del equipo (palabras en minúscula)
GAZETTEER = {
    "LOC": {"cartagena", "barranquilla", "bolívar", "getsemaní", "bocagrande", "manga", "turbaco"},
    "ORG": {"utb", "dian", "sena", "afinia", "ecopetrol", "icbf"},
}

def shape(w):
    s = re.sub(r"[A-ZÁÉÍÓÚÑÜ]", "X", w)
    s = re.sub(r"[a-záéíóúñü]", "x", s)
    s = re.sub(r"\d", "d", s)
    return re.sub(r"(.)\1+", r"\1\1", s)

def rasgos(toks, i, usar_gaz):
    w = toks[i]; lw = w.lower()
    f = {"bias": 1.0, "w": lw, "suf3": lw[-3:], "suf2": lw[-2:], "pre3": lw[:3], "shape": shape(w),
         "is_title": float(w.istitle()), "is_upper": float(w.isupper()), "has_digit": float(any(c.isdigit() for c in w))}
    if usar_gaz:
        for tipo, lista in GAZETTEER.items():
            if lw in lista: f[f"gaz:{tipo}"] = 1.0
    for d, nombre in ((-1, "-1"), (1, "+1")):
        j = i + d
        if 0 <= j < len(toks):
            f[f"{nombre}:w"] = toks[j].lower(); f[f"{nombre}:is_title"] = float(toks[j].istitle())
        else:
            f["BOS" if d < 0 else "EOS"] = 1.0
    return f

def entrenar_crf(train, usar_gaz, ruta_modelo, c1=0.1, c2=0.01, iters=100):
    tr = pycrfsuite.Trainer(verbose=False)
    for toks, tags in train:
        tr.append([rasgos(toks, i, usar_gaz) for i in range(len(toks))], tags)
    tr.set_params({"c1": c1, "c2": c2, "max_iterations": iters, "feature.possible_transitions": True})
    tr.train(ruta_modelo)
    tagger = pycrfsuite.Tagger(); tagger.open(ruta_modelo)
    return lambda sents_tokens: [tagger.tag([rasgos(t, i, usar_gaz) for i in range(len(t))]) for t in sents_tokens]

t0 = time.time()
MODELOS["CRF"] = entrenar_crf(train, False, "salidas/crf.crfsuite")
MODELOS["CRF + gazetteer"] = entrenar_crf(train, True, "salidas/crf_gaz.crfsuite")
print(f"CRF entrenados en {time.time() - t0:.0f} s")

### 3.3 Transformer afinado (opcional, `RUN_TRANSFORMER`)
BETO + capa de clasificación por token. Alineación: se etiqueta la **primera subpalabra** de cada palabra y el resto recibe −100. Sin GPU se usa un subconjunto (modo rápido).

In [ ]:
if RUN_TRANSFORMER:
    import torch
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments,
                              Trainer, DataCollatorForTokenClassification)
    MODELO_TR = "dccuchile/bert-base-spanish-wwm-cased"
    RAPIDO = not torch.cuda.is_available()
    etiquetas = ["O"] + [f"{p}-{t}" for t in TIPOS for p in ("B", "I")]
    label2id = {l: i for i, l in enumerate(etiquetas)}; id2label = {i: l for l, i in label2id.items()}
    tok = AutoTokenizer.from_pretrained(MODELO_TR)

    def a_dataset(sents):
        return Dataset.from_dict({"tokens": [t for t, _ in sents], "tags": [[label2id[x] for x in y] for _, y in sents]})

    def alinear(batch):
        enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
        enc["labels"] = []
        for b, labs in enumerate(batch["tags"]):
            prev, lab = None, []
            for wid in enc.word_ids(batch_index=b):
                lab.append(-100 if wid is None or wid == prev else labs[wid]); prev = wid
            enc["labels"].append(lab)
        return enc

    sub = train[:2000] if RAPIDO else train
    ds = a_dataset(sub).map(alinear, batched=True, remove_columns=["tokens", "tags"])
    modelo_tr = AutoModelForTokenClassification.from_pretrained(MODELO_TR, num_labels=len(etiquetas),
                    id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
    args = TrainingArguments(output_dir="salidas/beto", learning_rate=3e-5, per_device_train_batch_size=16,
                             num_train_epochs=1 if RAPIDO else 3, weight_decay=0.01, save_strategy="no",
                             logging_steps=50, report_to="none", seed=SEED)
    Trainer(model=modelo_tr, args=args, train_dataset=ds, processing_class=tok,
            data_collator=DataCollatorForTokenClassification(tok)).train()

    def predecir_tr(sents_tokens, bs=16):
        modelo_tr.eval(); salida = []
        for i in range(0, len(sents_tokens), bs):
            lote = sents_tokens[i:i + bs]
            enc = tok(lote, is_split_into_words=True, truncation=True, max_length=256, padding=True, return_tensors="pt")
            with torch.no_grad():
                pred = modelo_tr(**{k: v.to(modelo_tr.device) for k, v in enc.items()}).logits.argmax(-1).cpu().numpy()
            for b, toks in enumerate(lote):
                tags, vistos = ["O"] * len(toks), set()
                for j, wid in enumerate(enc.word_ids(batch_index=b)):
                    if wid is not None and wid not in vistos:
                        tags[wid] = id2label[int(pred[b, j])]; vistos.add(wid)
                salida.append(tags)
        return salida

    MODELOS["Transformer (BETO)" + (" rápido" if RAPIDO else "")] = predecir_tr
else:
    print("Transformer desactivado (RUN_TRANSFORMER = False).")

### 3.4 LLM local con pocos ejemplos (opcional, `RUN_LLM`)
El LLM recibe la especificación de la tarea y `K_SHOTS` ejemplos **del conjunto de entrenamiento** (nunca de la prueba). Devuelve JSON; el notebook alinea cada entidad con los tokens y cuenta lo que no se puede alinear.

**TODO:** mejoren el prompt (definiciones, reglas del dominio) y formulen **antes de ejecutar** una hipótesis sobre su efecto.

In [ ]:
K_SHOTS = 3
N_LLM = 40          # oraciones de prueba a etiquetar (el LLM es lento en CPU)

if RUN_LLM:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    llm_tok = AutoTokenizer.from_pretrained(LLM_ID)
    llm = AutoModelForCausalLM.from_pretrained(LLM_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                                               device_map="auto")

    SISTEMA = ("Eres un anotador experto de entidades nombradas en español. Tipos permitidos (y solo estos): "
               + "; ".join(f"{t} = {DESCRIPCIONES.get(t, t)}" for t in TIPOS)
               + ". Copia cada entidad EXACTAMENTE como aparece en la oración, token por token. No inventes entidades. "
               'Responde SOLO con JSON: {"entidades": [{"texto": "...", "tipo": "..."}]}. Si no hay, {"entidades": []}.')

    def json_oro(toks, tags):
        return json.dumps({"entidades": [{"texto": " ".join(toks[s:e]), "tipo": t} for s, e, t in bio_to_spans(tags)]},
                          ensure_ascii=False)

    # Ejemplos de entrenamiento con al menos dos entidades (deterministas)
    EJEMPLOS = [(t, g) for t, g in train if 8 <= len(t) <= 30 and len(bio_to_spans(g)) >= 2][:K_SHOTS]

    def mensajes(toks):
        m = [{"role": "system", "content": SISTEMA}]
        for t, g in EJEMPLOS:
            m += [{"role": "user", "content": "Oración: " + " ".join(t)}, {"role": "assistant", "content": json_oro(t, g)}]
        return m + [{"role": "user", "content": "Oración: " + " ".join(toks)}]

    diag_llm = Counter()
    def predecir_llm(sents_tokens):
        salida = []
        for toks in sents_tokens:
            texto = llm_tok.apply_chat_template(mensajes(toks), tokenize=False, add_generation_prompt=True)
            enc = llm_tok(texto, return_tensors="pt").to(llm.device)
            with torch.no_grad():
                out = llm.generate(**enc, max_new_tokens=256, do_sample=False)
            resp = llm_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
            labels = ["O"] * len(toks)
            m = re.search(r"\{.*\}", resp, re.S)
            try:
                ents = json.loads(m.group(0)).get("entidades", []) if m else None
            except (json.JSONDecodeError, AttributeError):
                ents = None
            if ents is None: diag_llm["json_invalido"] += 1; ents = []
            for e in ents:
                tipo = str(e.get("tipo", "")).upper().strip() if isinstance(e, dict) else ""
                et = str(e.get("texto", "")).split() if isinstance(e, dict) else []
                if tipo not in TIPOS: diag_llm["tipo_invalido"] += 1; continue
                for i in range(len(toks) - len(et) + 1):
                    if et and toks[i:i + len(et)] == et and all(l == "O" for l in labels[i:i + len(et)]):
                        labels[i] = "B-" + tipo
                        for j in range(i + 1, i + len(et)): labels[j] = "I-" + tipo
                        diag_llm["alineada"] += 1; break
                else:
                    diag_llm["no_alineable"] += 1
            salida.append(labels)
        return salida

    MODELOS[f"LLM {LLM_ID.split('/')[-1]} {K_SHOTS}-shot"] = predecir_llm
else:
    print("LLM desactivado (RUN_LLM = False).")

## Fase 4 · Evaluación
Todos los modelos se evalúan con las mismas oraciones. El LLM solo se evalúa en las primeras `N_LLM` oraciones; para compararlo con justicia, la tabla incluye también el F1 de los demás modelos **en esas mismas oraciones**.

In [ ]:
def evaluar(nombre, predecir, datos, n=None, intervalo=True):
    datos = datos if n is None else datos[:n]
    t0 = time.time(); pred = predecir([t for t, _ in datos]); seg = (time.time() - t0) / max(1, len(datos))
    oro = [g for _, g in datos]
    rep = entity_prf(oro, pred)
    lo, hi = bootstrap_f1(oro, pred, B=500) if intervalo else (float("nan"), float("nan"))
    por_tipo = rep.drop(index="micro")
    peor = por_tipo["F1"].idxmin() if len(por_tipo) else "-"
    return rep, pred, {"F1": rep.loc["micro", "F1"], "IC95": f"[{lo:.2f}, {hi:.2f}]", "tipo_peor": peor, "seg_por_oración": seg}

filas, PRED, REPORTES = [], {}, {}
es_llm = lambda nombre: nombre.startswith("LLM")
for nombre, f in MODELOS.items():
    n = N_LLM if es_llm(nombre) else None
    fila = {"modelo": nombre, "n_prueba": len(test_fuera) if n is None else min(n, len(test_fuera))}
    if test_dominio is not None and not es_llm(nombre):
        rep_d, _, r = evaluar(nombre, f, test_dominio, intervalo=False)
        fila["F1 en dominio"] = r["F1"]
    rep, pred, r = evaluar(nombre, f, test_fuera, n)
    PRED[nombre], REPORTES[nombre] = pred, rep
    fila.update({"F1 fuera de dominio": r["F1"], "IC95": r["IC95"], "tipo peor": r["tipo_peor"], "seg/oración": r["seg_por_oración"]})
    if RUN_LLM and not es_llm(nombre):
        fila[f"F1 en las {N_LLM} del LLM"] = entity_prf([g for _, g in test_fuera[:N_LLM]], pred[:N_LLM]).loc["micro", "F1"]
    filas.append(fila)

resultados = pd.DataFrame(filas).set_index("modelo")
resultados["cumple F1"] = resultados["F1 fuera de dominio"] >= CRITERIO_F1
resultados["cumple tiempo"] = resultados["seg/oración"] <= CRITERIO_SEG
resultados.round(3)

In [ ]:
if RUN_LLM:
    print("Diagnóstico del LLM:", dict(diag_llm))
# F1 por tipo del mejor modelo
mejor = resultados["F1 fuera de dominio"].idxmax()
print("Mejor modelo fuera de dominio:", mejor)
display(REPORTES[mejor].round(3))

### 4.1 Análisis por subgrupo
La función `grupo` asigna cada entidad de oro a un subgrupo. Por defecto separa entidades de 1 token y de varios. **TODO:** definan un corte propio del problema (por ejemplo nombres locales frente a extranjeros, entidades con sigla, entidades que no aparecen en entrenamiento).

In [ ]:
vistas_en_train = {" ".join(t[s:e]) for t, g in train for s, e, _ in bio_to_spans(g)}

def grupo(toks, span):
    # TODO: reemplazar o añadir un corte propio del problema
    s, e, tipo = span
    texto = " ".join(toks[s:e])
    return "vista en entrenamiento" if texto in vistas_en_train else "nueva"

def recall_por_grupo(datos, pred):
    c = defaultdict(lambda: [0, 0])
    for (toks, gold), p in zip(datos, pred):
        P = set(bio_to_spans(p))
        for sp in bio_to_spans(gold):
            g = grupo(toks, sp); c[g][1] += 1; c[g][0] += sp in P
    return pd.DataFrame([(g, a / t, t) for g, (a, t) in c.items()], columns=["grupo", "recall", "n"]).set_index("grupo")

recall_por_grupo(test_fuera, PRED[mejor]).round(3)

### 4.2 Análisis de errores
Se exporta un CSV con los errores del mejor modelo. **TODO:** clasifiquen a mano 10 errores (columna `causa`) con una hipótesis de por qué ocurrieron.

In [ ]:
errores = error_breakdown(test_fuera, PRED[mejor])
display(errores.clase_error.value_counts())
errores["texto_oración"] = errores["oración"].map(lambda k: " ".join(test_fuera[k][0]))
errores["causa"] = ""
errores.to_csv(f"salidas/errores_equipo_{EQUIPO}.csv", index=False, encoding="utf-8")
errores.head(10)

**✍️ Responder:** ¿el mejor modelo cumple los criterios de éxito? Decisión: **desplegar / iterar / descartar**, justificada con la tabla, el intervalo, los errores y el registro de riesgos.

## Fase 5 · Despliegue (lienzo)

**✍️ Responder:**

| Campo | Respuesta del equipo |
|---|---|
| Usuario y flujo | |
| Modo (lote o tiempo real) | |
| Hardware y costo por documento (usar `seg/oración`) | |
| Revisión humana (qué casos) | |
| Estrategia de salida (piloto, sombra, gradual) | |
| Privacidad (qué datos salen) | |

## Fase 6 · Monitoreo y mantenimiento
Una señal de alarma temprana es que los datos nuevos no se parezcan a los de entrenamiento. Esta celda compara la distribución de tipos y la tasa de palabras desconocidas entre entrenamiento y la muestra del equipo: es el tipo de indicador que se vigilaría en producción.

In [ ]:
def distribucion(sents):
    c = Counter(t for _, g in sents for _, _, t in bio_to_spans(g)); n = sum(c.values()) or 1
    return {k: v / n for k, v in c.items()}

vocab_train = {w for t, _ in train for w in t}
oov = lambda sents: sum(w not in vocab_train for t, _ in sents for w in t) / max(1, sum(len(t) for t, _ in sents))
deriva = pd.DataFrame({"entrenamiento": distribucion(train), "muestra del equipo": distribucion(test_fuera)}).fillna(0)
deriva["diferencia"] = deriva["muestra del equipo"] - deriva["entrenamiento"]
display(deriva.round(3))
if test_dominio:
    print(f"Palabras desconocidas en la prueba en dominio: {oov(test_dominio):.1%}")
print(f"Palabras desconocidas en la muestra del equipo: {oov(test_fuera):.1%}")

**✍️ Responder:** plan de monitoreo (indicador, cómo se mide, umbral de alerta, acción), responsable y frecuencia de reentrenamiento.

| Indicador | Cómo se mide | Umbral de alerta | Acción |
|---|---|---|---|
| | | | |

## Cierre · Exportar resultados para el documento

In [ ]:
resultados.round(3).to_csv(f"salidas/resultados_equipo_{EQUIPO}.csv", encoding="utf-8")
ficha_datos.to_csv(f"salidas/ficha_datos_equipo_{EQUIPO}.csv", index=False, encoding="utf-8")
riesgos.to_csv(f"salidas/riesgos_equipo_{EQUIPO}.csv", index=False, encoding="utf-8")
print("Versiones:", versiones)
print("DEMO activa: " + ("SÍ — reemplazar por los datos del equipo antes de entregar" if DEMO else "no"))
resultados.round(3)

### Lista de verificación antes de entregar
- [ ] `DEMO` = no (se usan los datos del equipo).
- [ ] Kernel → Restart & Run All corre sin errores.
- [ ] Registro de riesgos con al menos tres riesgos y sus mitigaciones.
- [ ] Gazetteer propio completado y su efecto comparado (CRF vs CRF + gazetteer).
- [ ] Corte por subgrupo propio en la sección 4.1.
- [ ] 10 errores clasificados con causa en el CSV.
- [ ] Todas las celdas **✍️ Responder** completas.
- [ ] Sin claves de API ni datos personales reales.